# Moshi × Ditto Streaming — A1 RunPod Notebook

**Real-time talking-head generation** powered by [Moshi](https://github.com/kyutai-labs/moshi) (speech LM) bridged to [Ditto](https://github.com/digital-avatar/ditto-talkinghead)

---
## § 0 — Repository

Clone the project and switch into its directory.

In [ ]:
!git clone https://github.com/MoshiHead/moshi-ditto-streaming-v2.git
%cd /workspace/moshi-ditto-streaming-v2

---
## § 1 — Install Dependencies

In [ ]:
# ── 1a: Project install script ─────────────────────────────────────────────────
import os, subprocess, sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
print(f"📁 Project root: {PROJECT_ROOT}")

install_script = os.path.join(PROJECT_ROOT, "install.sh")
if not os.path.isfile(install_script):
    raise FileNotFoundError(f"install.sh not found at: {install_script}")

print("🚀 Starting installation...\n")
result = subprocess.run(["bash", install_script], cwd=PROJECT_ROOT, text=True)
if result.returncode != 0:
    print("❌ Installation failed — check output above.")
else:
    print("\n✅ Installation complete!")

In [ ]:
# # ── 1b: System packages (ffmpeg + cuDNN) ───────────────────────────────────────
# !apt update -qq
# !apt-get install -y ffmpeg libcudnn8 libcudnn8-dev 2>&1 | tail -5

In [ ]:
!sed -i 's|http://archive.ubuntu.com/ubuntu|http://mirror.math.princeton.edu/pub/ubuntu|g' /etc/apt/sources.list
!apt-get update
!apt-get install -y ffmpeg libcudnn8 libcudnn8-dev 2>&1 | tail -5

In [ ]:
# ── 1c: TurboJPEG (optional, auto-detected at runtime) ─────────────────────────
# Falls back to cv2 → PIL if not available.
!pip install -q PyTurboJPEG 2>&1 | tail -3
try:
    from turbojpeg import TurboJPEG
    print("✅ TurboJPEG available — fastest JPEG encoding enabled")
except ImportError:
    print("ℹ️  TurboJPEG not available — will use cv2 or PIL (still functional)")

In [ ]:
# ── 1d: TensorRT + cuda-python (required for Ditto) ────────────────────────────
!pip install -q filetype cython huggingface_hub
!pip install "turbojpeg<2"
!pip install -q cuda-python==12.4.0
!python -c "from cuda import cuda, cudart, nvrtc; print('✅ cuda-python OK')"
!pip install -q tensorrt==8.6.1 --extra-index-url https://pypi.nvidia.com
!python -c "import tensorrt as trt; print('✅ TRT', trt.__version__)"

In [ ]:
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-500:])
    return r.returncode

print('Step 1/3 — Upgrading typing_extensions to >=4.6.0 ...')
rc = run([sys.executable, '-m', 'pip', 'install', '-q',
          'typing_extensions>=4.6.0', '--upgrade'])
print('✅ done' if rc == 0 else '❌ failed')

print('Step 2/3 — Reinstalling FastAPI with compatible version ...')
rc = run([sys.executable, '-m', 'pip', 'install', '-q',
          'fastapi>=0.100.0', '--upgrade', '--force-reinstall', '--no-deps'])
print('✅ done' if rc == 0 else '❌ failed')

run([sys.executable, '-m', 'pip', 'install', '-q',
     'starlette>=0.27.0', '--upgrade'])

print('Step 3/3 — Verifying fix ...')
try:
    import sys as _sys
    for mod in list(_sys.modules.keys()):
        if 'fastapi' in mod or 'typing_extensions' in mod or 'starlette' in mod:
            del _sys.modules[mod]
    import fastapi
    print(f'✅ FastAPI {fastapi.__version__} imports successfully!')
except Exception as e:
    print(f'❌ Still failing: {e}')
    print('   → Try restarting the kernel and re-running this cell.')

---
## § 2 — Verify Environment

In [ ]:
import torch, shutil, sys

print("=" * 58)
print("  Environment Check")
print("=" * 58)
print(f"  Python  : {sys.version.split()[0]}")
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA ok : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print()
for pkg in ['fastapi', 'uvicorn', 'onnxruntime']:
    try:
        m = __import__(pkg)
        print(f"  {pkg:<14}: {m.__version__}")
    except ImportError as e:
        print(f"  {pkg:<14}: ❌ {e}")

# JPEG encoder priority check
jpeg_backend = "PIL (slowest)"
try:
    from turbojpeg import TurboJPEG; jpeg_backend = "TurboJPEG (fastest)"
except ImportError:
    try:
        import cv2; jpeg_backend = "OpenCV cv2 (fast)"
    except ImportError:
        pass

print(f"  JPEG enc        : {jpeg_backend}")
print(f"  FFmpeg  : {'✅' if shutil.which('ffmpeg') else '❌ NOT FOUND'}")
print("=" * 58)

---
## § 3 — Download Checkpoints

In [ ]:
from huggingface_hub import snapshot_download
import os

PROJECT_ROOT = os.path.abspath(os.getcwd())
CKPT_DIR     = os.path.join(PROJECT_ROOT, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Bridge .pt checkpoint (streaming, KV-cache) ────────────────────────────────
print("📥 Downloading bridge_best.pt...")
snapshot_download(
    repo_id="Darknsu/bridge_best_onnx",
    repo_type="dataset",
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False,
)
print("✅ bridge_best.pt downloaded.")

# ── Bridge .onnx checkpoint (max throughput, no KV-cache) ─────────────────────
print("\n📥 Downloading bridge_best.onnx...")
snapshot_download(
    repo_id="Darknsu/librispeech-full-dataset-model",
    repo_type="dataset",
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False,
    allow_patterns=["bridge_best.pt"],
)
print("✅ bridge_best.onnx downloaded.")

# ── Ditto TRT models ───────────────────────────────────────────────────────────
print("\n📥 Downloading Ditto TRT models (may take a few minutes)...")
snapshot_download(
    repo_id="digital-avatar/ditto-talkinghead",
    repo_type="model",
    local_dir=os.path.join(PROJECT_ROOT, "ditto-inference", "checkpoints"),
    local_dir_use_symlinks=False,
)
print("✅ Ditto models downloaded.")

---
## § 4 — Configure Server

In [ ]:
import os, torch

PROJECT_ROOT = os.path.abspath(os.getcwd())
print(f"📁 Project root: {PROJECT_ROOT}")

# ══════════════════════════════════════════════════════
# Server
# ══════════════════════════════════════════════════════
SERVER_HOST = "0.0.0.0"
SERVER_PORT = 7860

# ══════════════════════════════════════════════════════
# Moshi (speech LM)
# ══════════════════════════════════════════════════════
MOSHI_HF_REPO   = "kyutai/moshiko-pytorch-q8"
MOSHI_WEIGHT    = None   # set to a local path to skip HF download
MIMI_WEIGHT     = None
MOSHI_TOKENIZER = None

# ══════════════════════════════════════════════════════
# Bridge
# ══════════════════════════════════════════════════════
BRIDGE_CKPT   = os.path.join(PROJECT_ROOT, "checkpoints", "bridge_best.pt")  # .pt streaming
BRIDGE_CONFIG = os.path.join(PROJECT_ROOT, "bridge_module", "config.yaml")

# Number of Mimi tokens to batch before running the bridge.
# Recommended: 2 on A100 for lowest latency.
BRIDGE_CHUNK = 2

# Max wait (ms) before flushing a partial token batch to Ditto.
# 50 ms = flush at most 50 ms after the first token of a new batch arrives.
BRIDGE_FLUSH_TIMEOUT_MS = 50

# ══════════════════════════════════════════════════════
# Ditto (TRT talking-head renderer)
# ══════════════════════════════════════════════════════
DITTO_DATA_ROOT = os.path.join(PROJECT_ROOT, "ditto-inference",
                               "checkpoints", "ditto_trt_Ampere_Plus")
DITTO_CFG_PKL   = os.path.join(PROJECT_ROOT, "ditto-inference",
                               "checkpoints", "ditto_cfg",
                               "v0.4_hubert_cfg_trt.pkl")
DITTO_EMO            = 4     # emotion index 0–7
DITTO_SAMPLING_STEPS = 10    # lower = faster; try 10–20 for speed vs. quality
DITTO_JPEG_QUALITY   = 70    # lower = smaller payload, faster transfer (70 is optimal for real-time)

# ══════════════════════════════════════════════════════
# Path validation
# ══════════════════════════════════════════════════════
print("\n📋 Path Check")
print("─" * 60)
errors = []
checks = [
    ("Bridge .pt",      BRIDGE_CKPT),
    ("Bridge config",   BRIDGE_CONFIG),
    ("Ditto data root", DITTO_DATA_ROOT),
    ("Ditto cfg pkl",   DITTO_CFG_PKL),
]
for label, path in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'}  {label:<22} {path}")
    if not exists:
        errors.append(label)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n  Device          : {DEVICE}")
print(f"  Bridge chunk    : {BRIDGE_CHUNK} tokens ({BRIDGE_CHUNK * 80} ms audio)")
print(f"  Bridge flush    : {BRIDGE_FLUSH_TIMEOUT_MS} ms adaptive timeout")
print(f"  Ditto steps     : {DITTO_SAMPLING_STEPS} (lower = faster)")

if errors:
    print(f"\n⚠️  {len(errors)} missing path(s): {errors}")
else:
    print("\n✅ All checks passed — ready to launch!")

---
## § 5 — Pipeline Import Health Check

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.getcwd())
for p in [PROJECT_ROOT,
          os.path.join(PROJECT_ROOT, 'bridge_module'),
          os.path.join(PROJECT_ROOT, 'moshi-inference')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("🔬 Pipeline Import Health Check")
print("═" * 50)

checks = {
    "pipeline.sync_types":           "TaggedToken, TaggedFeatures, TaggedFrame, seq_pack",
    "pipeline.streaming_moshi":      "StreamingMoshiState",
    "pipeline.ditto_stream_adapter": "DittoStreamAdapter",
    "inference":                     "StreamingBridgeInference",
}

all_ok = True
for mod, symbols in checks.items():
    try:
        m = __import__(mod, fromlist=symbols.split(', '))
        for sym in symbols.split(', '):
            getattr(m, sym.strip())
        print(f"  ✅  {mod:<40}  [{symbols}]")
    except Exception as e:
        print(f"  ❌  {mod:<40}  ERROR: {e}")
        all_ok = False

# Verify seq_pack struct encoding
try:
    import struct
    b = struct.pack('>I', 12345)
    assert len(b) == 4
    print(f"  ✅  seq_pack (struct.pack >I)              [4-byte big-endian uint32]")
except Exception as e:
    print(f"  ❌  seq_pack check failed: {e}")
    all_ok = False

print("═" * 50)
print("\n✅ All imports OK — safe to launch server." if all_ok else
      "\n❌ Fix import errors above before launching.")

---
## § 6 — Launch Server

In [ ]:
# ── 6a: Kill any existing server process ───────────────────────────────────────
import os, signal, time

def get_pids_on_port(port):
    """Return PIDs of processes listening on `port` by scanning /proc/net/tcp."""
    hex_port = format(port, '04X')
    inodes = set()
    for path in ["/proc/net/tcp", "/proc/net/tcp6"]:
        try:
            with open(path) as f:
                for line in f.readlines()[1:]:
                    parts = line.split()
                    if len(parts) > 9 and parts[1].split(":")[1] == hex_port:
                        inodes.add(parts[9])
        except FileNotFoundError:
            pass
    pids = []
    for pid in os.listdir("/proc"):
        if not pid.isdigit():
            continue
        try:
            for fd in os.listdir(f"/proc/{pid}/fd"):
                try:
                    link = os.readlink(f"/proc/{pid}/fd/{fd}")
                    if "socket:[" in link and link.split("[")[1].rstrip("]") in inodes:
                        pids.append(int(pid))
                        break
                except OSError:
                    pass
        except OSError:
            pass
    return pids


print("🔪 Killing any existing server processes...")
killed = []
for pid in os.listdir("/proc"):
    if not pid.isdigit():
        continue
    try:
        with open(f"/proc/{pid}/cmdline") as f:
            if "streaming_server.py" in f.read():
                os.kill(int(pid), signal.SIGKILL)
                killed.append(pid)
    except (OSError, PermissionError):
        pass

if killed:
    print(f"   💀 Killed PIDs: {', '.join(killed)}")
time.sleep(1)

remaining = get_pids_on_port(SERVER_PORT)
for pid in remaining:
    try:
        os.kill(pid, signal.SIGKILL)
        print(f"   💀 Killed port-holder PID {pid}")
    except Exception:
        pass
if remaining:
    time.sleep(1)

if get_pids_on_port(SERVER_PORT):
    print(f"⚠️  Port {SERVER_PORT} still busy!")
else:
    print(f"✅ Port {SERVER_PORT} is free — safe to launch.")

In [ ]:
# ── 6b: Start the streaming server ────────────────────────────────────────────
# Interrupt the kernel (■) to stop the server.
!python streaming_server.py